# V10 HGP concordance inspection vs PINN

This short notebook inspects the finished hierarchical GP-style concordance product and compares it with the existing PINN concordance fields.

**What this checks**

1. HGP run metadata, holdout calibration and per-band field amplitudes from the JSON sidecar.
2. HGP mean field vs the PINN head-residual field on a common sky grid.
3. HGP posterior uncertainty and coverage maps.
4. The HGP common / Rubin-or-NISP / band-specific decomposition.

The grids are not compared pixel-by-pixel directly: the HGP product is written on a 5 arcsec mesh, while the saved PINN products are on a 1 arcsec mesh. We sample the PINN field at the HGP sky coordinates before comparing vectors.

**Important caveat.** The default comparison below is now apples-to-apples in source catalogue: HGP and PINN both use the CenterNet anchor cache. HGP fits `head_resid`; the default PINN head-residual product was refit from the same `anchors_centernet_v10.npz` cache for this notebook. Differences between HGP and PINN are therefore mainly model/prior differences, with a small remaining difference from each fitter's clipping/weighting choices.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from scipy.ndimage import map_coordinates

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 180,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

ROOT = Path.cwd().parent if Path.cwd().name == 'io' else Path.cwd()
import os
RUN_KEY = os.environ.get('JAISP_ASTROMETRY_RUN', 'no_psf')
RUNS = {
    'no_psf': {
        'ckpt': ROOT / 'models/checkpoints/latent_position_v10_no_psf',
        'anchors': 'anchors_centernet_v10.npz',
    },
    'epsf_centroid': {
        'ckpt': ROOT / 'models/checkpoints/latent_position_v10_epsf_centroid',
        'anchors': 'anchors_centernet_v10_epsf.npz',
    },
}
if RUN_KEY not in RUNS:
    raise ValueError(f"Unknown JAISP_ASTROMETRY_RUN={RUN_KEY!r}; choose one of {sorted(RUNS)}")
CKPT = RUNS[RUN_KEY]['ckpt']

HGP_FITS = CKPT / 'concordance_hgp_head_resid_dedup.fits'
HGP_JSON = CKPT / 'concordance_hgp_head_resid_dedup.fits.json'
PINN_HEAD_FITS = CKPT / 'concordance_pinn_centernet_v10_head_resid.fits'
PINN_RAW_FITS = CKPT / 'concordance_pinn_centernet_v10_raw.fits'

BANDS = ['U', 'G', 'R', 'I', 'Z', 'Y', 'NISP_Y', 'NISP_J', 'NISP_H']
BAND_LABEL = {
    'U': 'u', 'G': 'g', 'R': 'r', 'I': 'i', 'Z': 'z', 'Y': 'y',
    'NISP_Y': 'NISP Y', 'NISP_J': 'NISP J', 'NISP_H': 'NISP H',
}

# COVERAGE in these FITS files is distance to the nearest anchor, in arcsec.
# Use a support mask for quantitative comparisons so edge extrapolation does
# not dominate field RMS. Tune these thresholds if you want a stricter mask.
MAX_NEAREST_ANCHOR_ARCSEC = 30.0
MAX_HGP_STD_RADIAL_MAS = 10.0

for path in [HGP_FITS, HGP_JSON, PINN_HEAD_FITS, PINN_RAW_FITS]:
    if not path.exists():
        raise FileNotFoundError(path)

print('HGP:', HGP_FITS.relative_to(ROOT))
print('PINN head residual:', PINN_HEAD_FITS.relative_to(ROOT))
print('PINN raw:', PINN_RAW_FITS.relative_to(ROOT))

In [ ]:
def product_header_summary(path, band='I'):
    with fits.open(path, memmap=True) as hdul:
        primary = hdul[0].header
        h = hdul[f'{band}.DRA'].header
        return {
            'product': path.name,
            'method_primary': primary.get('METHOD', ''),
            'method_band': h.get('METHOD', ''),
            'anchors': primary.get('ANCHORS', ''),
            'offset_kind': primary.get('OFFKIND', ''),
            'n_anchor_primary': primary.get('NANCHOR', np.nan),
            'n_anchor_band': h.get('NANCHORS', h.get('NSRC', np.nan)),
            'dstep_arcsec': h.get('DSTEP', primary.get('DSTEP', np.nan)),
            'shape': (h['NAXIS2'], h['NAXIS1']),
            'ra0': h.get('RA0', h.get('CRVAL1', np.nan)),
            'dec0': h.get('DEC0', h.get('CRVAL2', np.nan)),
        }


product_df = pd.DataFrame([
    product_header_summary(HGP_FITS),
    product_header_summary(PINN_HEAD_FITS),
    product_header_summary(PINN_RAW_FITS),
])
display(product_df)

## 1. HGP run summary

The sidecar JSON is the quickest sanity check: it records which anchors were used, which residual was fitted, what length scales were available, and whether the posterior uncertainty was calibrated on a spatial holdout.

In [ ]:
with HGP_JSON.open() as f:
    hgp_meta = json.load(f)

args = hgp_meta['args']
train = hgp_meta['train']
holdout = hgp_meta.get('holdout', {})
basis = hgp_meta['basis']

run_summary = pd.Series({
    'anchors': args['anchors'],
    'offset_kind': args['offset_kind'],
    'pool': args['pool'],
    'n_anchors': hgp_meta['n_anchors'],
    'n_features': hgp_meta['hierarchy']['n_features'],
    'length_scales_arcsec': args['length_scales'],
    'mesh_shape': tuple(hgp_meta['output']['mesh_shape']),
    'dstep_arcsec': args['dstep_arcsec'],
    'train_resid_med_mas': train['train_resid_med_mas'],
    'holdout_resid_med_mas': holdout.get('holdout_resid_med_mas', np.nan),
    'holdout_z_std_dra': holdout.get('z_std_dra', np.nan),
    'holdout_z_std_ddec': holdout.get('z_std_ddec', np.nan),
    'uncertainty_calibration_factor': holdout.get('uncertainty_calibration_factor', np.nan),
})
display(run_summary.to_frame('value'))

basis_df = pd.DataFrame(basis['scales'])
display(basis_df)

In [ ]:
band_rows = []
for band, vals in hgp_meta['output']['bands'].items():
    band_rows.append({
        'band': band,
        'n_anchors': vals['n_anchors'],
        'hgp_field_rms_mas_json': vals['field_rms_mas'],
        'hgp_median_std_mas_json': vals['median_std_mas'],
    })
hgp_band_json = pd.DataFrame(band_rows)
display(hgp_band_json)

fig, ax = plt.subplots(figsize=(7.2, 3.2))
x = np.arange(len(hgp_band_json))
ax.bar(x - 0.18, hgp_band_json['hgp_field_rms_mas_json'], width=0.36, label='HGP field RMS')
ax.bar(x + 0.18, hgp_band_json['hgp_median_std_mas_json'], width=0.36, label='HGP median posterior std')
ax.set_xticks(x, hgp_band_json['band'], rotation=30, ha='right')
ax.set_ylabel('mas')
ax.set_title('HGP sidecar per-band summary')
ax.legend(frameon=False)
fig.tight_layout()

## 2. Compare HGP and PINN on the same sky grid

For each band, use the HGP WCS as the target grid and interpolate the PINN maps at those exact sky positions. All FITS values are in arcsec; the summary table reports milliarcseconds.

In [ ]:
def grid_world_from_header(header):
    h = int(header['NAXIS2'])
    w = int(header['NAXIS1'])
    yy, xx = np.mgrid[:h, :w]
    ra, dec = WCS(header).wcs_pix2world(xx, yy, 0)
    return ra, dec


def sample_image_at_world(hdu, ra, dec, order=1):
    data = np.asarray(hdu.data, dtype=np.float64)
    x, y = WCS(hdu.header).wcs_world2pix(ra, dec, 0)
    sampled = map_coordinates(
        data,
        [y.ravel(), x.ravel()],
        order=order,
        mode='constant',
        cval=np.nan,
    ).reshape(ra.shape)
    return sampled


def amp_mas(dra_arcsec, dde_arcsec):
    return np.hypot(dra_arcsec, dde_arcsec) * 1000.0


def rms_amp_mas(dra_arcsec, dde_arcsec, mask=None):
    a = amp_mas(dra_arcsec, dde_arcsec)
    if mask is not None:
        a = a[mask]
    return float(np.sqrt(np.nanmean(a ** 2))) if np.isfinite(a).any() else np.nan


def median_amp_mas(dra_arcsec, dde_arcsec, mask=None):
    a = amp_mas(dra_arcsec, dde_arcsec)
    if mask is not None:
        a = a[mask]
    return float(np.nanmedian(a)) if np.isfinite(a).any() else np.nan


def load_resampled_band(band):
    with fits.open(HGP_FITS, memmap=True) as hgp, \
         fits.open(PINN_HEAD_FITS, memmap=True) as pinn_head, \
         fits.open(PINN_RAW_FITS, memmap=True) as pinn_raw:
        h_hdu = hgp[f'{band}.DRA']
        ra, dec = grid_world_from_header(h_hdu.header)
        out = {
            'ra': ra,
            'dec': dec,
            'hgp_dra': np.asarray(hgp[f'{band}.DRA'].data, dtype=np.float64),
            'hgp_dde': np.asarray(hgp[f'{band}.DDE'].data, dtype=np.float64),
            'hgp_dra_std': np.asarray(hgp[f'{band}.DRA_STD'].data, dtype=np.float64),
            'hgp_dde_std': np.asarray(hgp[f'{band}.DDE_STD'].data, dtype=np.float64),
            'hgp_coverage': np.asarray(hgp['COVERAGE'].data, dtype=np.float64) if 'COVERAGE' in hgp else np.ones_like(ra),
            'pinn_head_dra': sample_image_at_world(pinn_head[f'{band}.DRA'], ra, dec),
            'pinn_head_dde': sample_image_at_world(pinn_head[f'{band}.DDE'], ra, dec),
            'pinn_head_coverage': sample_image_at_world(pinn_head['COVERAGE'], ra, dec) if 'COVERAGE' in pinn_head else np.ones_like(ra),
            'pinn_raw_dra': sample_image_at_world(pinn_raw[f'{band}.DRA'], ra, dec),
            'pinn_raw_dde': sample_image_at_world(pinn_raw[f'{band}.DDE'], ra, dec),
        }
    return out


def finite_mask(d):
    finite = np.isfinite(d['hgp_dra']) & np.isfinite(d['hgp_dde'])
    finite &= np.isfinite(d['pinn_head_dra']) & np.isfinite(d['pinn_head_dde'])
    finite &= np.isfinite(d['pinn_raw_dra']) & np.isfinite(d['pinn_raw_dde'])
    finite &= np.isfinite(d['hgp_coverage']) & np.isfinite(d['pinn_head_coverage'])
    return finite


def support_mask(d, max_anchor_dist_arcsec=MAX_NEAREST_ANCHOR_ARCSEC, max_hgp_std_mas=MAX_HGP_STD_RADIAL_MAS):
    mask = finite_mask(d)
    mask &= d['hgp_coverage'] <= max_anchor_dist_arcsec
    mask &= d['pinn_head_coverage'] <= max_anchor_dist_arcsec
    if max_hgp_std_mas is not None:
        mask &= amp_mas(d['hgp_dra_std'], d['hgp_dde_std']) <= max_hgp_std_mas
    return mask

In [ ]:
rows = []
for band in BANDS:
    d = load_resampled_band(band)
    m_full = finite_mask(d)
    m = support_mask(d)
    diff_dra = d['hgp_dra'] - d['pinn_head_dra']
    diff_dde = d['hgp_dde'] - d['pinn_head_dde']
    std_radial = amp_mas(d['hgp_dra_std'], d['hgp_dde_std'])
    rows.append({
        'band': BAND_LABEL[band],
        'n_full': int(m_full.sum()),
        'n_support': int(m.sum()),
        'support_frac': float(m.sum() / max(m_full.sum(), 1)),
        'hgp_rms_full_mas': rms_amp_mas(d['hgp_dra'], d['hgp_dde'], m_full),
        'hgp_median_support_mas': median_amp_mas(d['hgp_dra'], d['hgp_dde'], m),
        'hgp_rms_support_mas': rms_amp_mas(d['hgp_dra'], d['hgp_dde'], m),
        'pinn_head_rms_support_mas': rms_amp_mas(d['pinn_head_dra'], d['pinn_head_dde'], m),
        'hgp_minus_pinn_rms_support_mas': rms_amp_mas(diff_dra, diff_dde, m),
        'hgp_median_std_support_mas': float(np.nanmedian(std_radial[m])),
        'pinn_raw_rms_support_mas': rms_amp_mas(d['pinn_raw_dra'], d['pinn_raw_dde'], m),
    })

compare_df = pd.DataFrame(rows)
display(compare_df.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
x = np.arange(len(compare_df))
ax.plot(x, compare_df['hgp_rms_full_mas'], marker='o', linestyle=':', label='HGP RMS, full grid')
ax.plot(x, compare_df['hgp_rms_support_mas'], marker='o', label='HGP RMS, support mask')
ax.plot(x, compare_df['pinn_head_rms_support_mas'], marker='o', label='PINN head-resid RMS, support')
ax.plot(x, compare_df['hgp_minus_pinn_rms_support_mas'], marker='o', label='|HGP - PINN| RMS, support')
ax.plot(x, compare_df['hgp_median_std_support_mas'], marker='o', linestyle='--', label='HGP median posterior std, support')
ax.set_xticks(x, compare_df['band'], rotation=30, ha='right')
ax.set_ylabel('mas')
ax.set_title('Field-level comparison on the HGP grid')
ax.legend(frameon=False, ncol=2)
fig.tight_layout()

## 3. Map view for one band

Use this as the visual sanity check. The selected band defaults to Rubin `i`, but any entry from `BANDS` above can be used.

In [ ]:
PLOT_BAND = 'I'
d = load_resampled_band(PLOT_BAND)
m_full = finite_mask(d)
m = support_mask(d)

maps = {
    'HGP |F| (full)': (amp_mas(d['hgp_dra'], d['hgp_dde']), m_full, 'mas'),
    'PINN head-resid |F| (support)': (amp_mas(d['pinn_head_dra'], d['pinn_head_dde']), m, 'mas'),
    '|HGP - PINN| (support)': (amp_mas(d['hgp_dra'] - d['pinn_head_dra'], d['hgp_dde'] - d['pinn_head_dde']), m, 'mas'),
    'HGP posterior std (full)': (amp_mas(d['hgp_dra_std'], d['hgp_dde_std']), m_full, 'mas'),
    'Nearest anchor dist': (d['hgp_coverage'], m_full, 'arcsec'),
    'Support mask': (m.astype(float), m_full, 'mask'),
}

fig, axes = plt.subplots(2, 3, figsize=(10.5, 8.4), constrained_layout=True)
for ax, (title, (image, mask, unit)) in zip(axes.ravel(), maps.items()):
    show = np.asarray(image, dtype=np.float64).copy()
    show[~mask] = np.nan
    if unit == 'arcsec':
        vmin, vmax, cmap = 0, np.nanpercentile(show, 99), 'viridis'
        label = 'nearest anchor [arcsec]'
    elif unit == 'mask':
        vmin, vmax, cmap = 0, 1, 'gray_r'
        label = 'support'
    else:
        vmin, vmax, cmap = 0, np.nanpercentile(show, 98), 'magma'
        label = 'mas'
    im = ax.imshow(show, origin='lower', cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02, label=label)

fig.suptitle(f'{BAND_LABEL[PLOT_BAND]}: HGP vs PINN on the HGP grid', y=1.02)

In [ ]:
rng = np.random.default_rng(42)
idx = np.flatnonzero(m.ravel())
idx = rng.choice(idx, size=min(8000, idx.size), replace=False)

hgp_dra = d['hgp_dra'].ravel()[idx] * 1000
hgp_dde = d['hgp_dde'].ravel()[idx] * 1000
pinn_dra = d['pinn_head_dra'].ravel()[idx] * 1000
pinn_dde = d['pinn_head_dde'].ravel()[idx] * 1000

lim = np.nanpercentile(np.abs(np.r_[hgp_dra, hgp_dde, pinn_dra, pinn_dde]), 99)
lim = max(lim, 1.0)

fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.4), constrained_layout=True)
for ax, hgp_val, pinn_val, name in [
    (axes[0], hgp_dra, pinn_dra, r'$\Delta$RA*'),
    (axes[1], hgp_dde, pinn_dde, r'$\Delta$Dec'),
]:
    ax.scatter(pinn_val, hgp_val, s=3, alpha=0.25, rasterized=True)
    ax.plot([-lim, lim], [-lim, lim], color='black', lw=1)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_xlabel(f'PINN {name} [mas]')
    ax.set_ylabel(f'HGP {name} [mas]')
    ax.set_title(name)
fig.suptitle(f'{BAND_LABEL[PLOT_BAND]} vector-component agreement')

## 4. HGP hierarchical decomposition

This is the main thing PINN did not give us directly: a split between common, instrument-group and band-specific smooth structure. The numbers below are field RMS values from the HGP output sidecar.

In [ ]:
component_df = (
    pd.DataFrame([
        {'component': name, 'field_rms_mas': vals['field_rms_mas']}
        for name, vals in hgp_meta['output'].get('components', {}).items()
    ])
    .sort_values('field_rms_mas', ascending=True)
)
display(component_df)

fig, ax = plt.subplots(figsize=(7, max(3.5, 0.22 * len(component_df))))
ax.barh(component_df['component'], component_df['field_rms_mas'])
ax.set_xlabel('field RMS [mas]')
ax.set_title('HGP component amplitudes')
fig.tight_layout()

## 5. Anchor-level zero-field sanity check

A smooth field should improve the residuals at the actual anchors, not only make a visually structured map. This cell samples HGP and PINN at the CenterNet head-residual anchor positions and compares both to the zero-field baseline. If the median residual barely changes, the fitted smooth field is below the source-level noise floor.

In [ ]:
ANCHORS = CKPT / RUNS[RUN_KEY]['anchors']
BAND_KEY = {'U': 'u', 'G': 'g', 'R': 'r', 'I': 'i', 'Z': 'z', 'Y': 'y',
            'NISP_Y': 'nisp_Y', 'NISP_J': 'nisp_J', 'NISP_H': 'nisp_H'}


def sample_field_pair(hdul, band, ra, dec, suffix=''):
    prefix = band
    dra = sample_image_at_world(hdul[f'{prefix}.DRA{suffix}'], ra, dec)
    dde = sample_image_at_world(hdul[f'{prefix}.DDE{suffix}'], ra, dec)
    return np.stack([dra, dde], axis=1)


anchor_rows = []
anchors = np.load(ANCHORS, allow_pickle=True)
with fits.open(HGP_FITS, memmap=True) as hgp, fits.open(PINN_HEAD_FITS, memmap=True) as pinn:
    for band in BANDS:
        key = BAND_KEY[band]
        ra = anchors[f'{key}_ra']
        dec = anchors[f'{key}_dec']
        off = anchors[f'{key}_head_resid'].astype(np.float64)
        hgp_pred = sample_field_pair(hgp, band, ra, dec)
        pinn_pred = sample_field_pair(pinn, band, ra, dec)
        hgp_std = sample_field_pair(hgp, band, ra, dec, suffix='_STD')
        hgp_dist = sample_image_at_world(hgp['COVERAGE'], ra, dec)
        pinn_dist = sample_image_at_world(pinn['COVERAGE'], ra, dec)
        finite = (
            np.isfinite(off).all(axis=1)
            & np.isfinite(hgp_pred).all(axis=1)
            & np.isfinite(pinn_pred).all(axis=1)
            & np.isfinite(hgp_std).all(axis=1)
            & np.isfinite(hgp_dist)
            & np.isfinite(pinn_dist)
        )
        finite &= amp_mas(off[:, 0], off[:, 1]) < 200.0
        support = finite & (hgp_dist <= MAX_NEAREST_ANCHOR_ARCSEC) & (pinn_dist <= MAX_NEAREST_ANCHOR_ARCSEC)
        support &= amp_mas(hgp_std[:, 0], hgp_std[:, 1]) <= MAX_HGP_STD_RADIAL_MAS
        for name, mask in [('finite', finite), ('support', support)]:
            if not mask.any():
                continue
            zero = amp_mas(off[mask, 0], off[mask, 1])
            after_hgp = amp_mas(off[mask, 0] - hgp_pred[mask, 0], off[mask, 1] - hgp_pred[mask, 1])
            after_pinn = amp_mas(off[mask, 0] - pinn_pred[mask, 0], off[mask, 1] - pinn_pred[mask, 1])
            anchor_rows.append({
                'band': BAND_LABEL[band],
                'mask': name,
                'n': int(mask.sum()),
                'zero_field_med_mas': float(np.nanmedian(zero)),
                'hgp_after_med_mas': float(np.nanmedian(after_hgp)),
                'pinn_after_med_mas': float(np.nanmedian(after_pinn)),
                'hgp_delta_med_mas': float(np.nanmedian(zero) - np.nanmedian(after_hgp)),
                'pinn_delta_med_mas': float(np.nanmedian(zero) - np.nanmedian(after_pinn)),
                'hgp_field_med_mas': median_amp_mas(hgp_pred[:, 0], hgp_pred[:, 1], mask),
                'pinn_field_med_mas': median_amp_mas(pinn_pred[:, 0], pinn_pred[:, 1], mask),
                'hgp_std_med_mas': median_amp_mas(hgp_std[:, 0], hgp_std[:, 1], mask),
                'nearest_anchor_p95_arcsec': float(np.nanpercentile(hgp_dist[mask], 95)),
            })

anchor_df = pd.DataFrame(anchor_rows)
display(anchor_df.round(3))

weighted = []
for mask_name, group in anchor_df.groupby('mask'):
    weights = group['n'].to_numpy()
    weighted.append({
        'mask': mask_name,
        'n_total': int(weights.sum()),
        'hgp_delta_source_weighted_mas': float(np.average(group['hgp_delta_med_mas'], weights=weights)),
        'pinn_delta_source_weighted_mas': float(np.average(group['pinn_delta_med_mas'], weights=weights)),
    })
display(pd.DataFrame(weighted).round(4))

## Reading the first result

- The HGP `COVERAGE` extension is a nearest-anchor distance map in arcsec. Large values mean weak support / extrapolation, not high anchor density.
- Compare the support-masked columns before interpreting field amplitudes. The full-grid HGP RMS is useful as an edge-extrapolation diagnostic, not as an astrometric correction claim.
- `|HGP - PINN| RMS` is the model-disagreement scale, not a new astrometric residual.
- `HGP posterior std` is the uncertainty of the fitted smooth field under the HGP model after holdout calibration.
- `PINN raw |F|` is context for the pre-head coherent field; the fair post-head comparison is HGP head-residual vs PINN head-residual.
- The HGP output grid is sampled every 5 arcsec on the sky. That is a spatial sampling interval, not a 5 mas accuracy floor. Sub-mas-valued corrections can be stored on a 5 arcsec grid if the fitted field is smooth on scales much larger than 5 arcsec.
- The current caution is not the grid size. The caution is the model comparison: the CenterNet PINN head-residual field is near 1 mas RMS, while HGP is several mas RMS and is dominated by the common component. Before using the HGP correction as the production field, inspect whether the HGP hierarchy/priors are absorbing residual source noise into a coherent common field.

## 6. Raw-anchor sanity check: HGP vs PINN before the head

The head-residual comparison above is only meaningful if the two solvers agree on what the smooth field looks like *before* the head touches it. The PINN raw fit (`concordance_pinn_centernet_v10_raw.fits`) recovers the ECDFS ~5-9 mas raw concordance. This section refits HGP on the same raw anchors and checks two things:

1. **Method-independence on raw**: do HGP and PINN find the same raw smooth field, on supported regions?
2. **Anchor-level signal on raw**: does subtracting either field improve raw `|offset|` residuals at the actual anchor positions, or are they tied with the zero-field baseline?

If HGP and PINN agree on raw and disagree on head_resid, the head-residual disagreement is genuinely about the post-head ~1 mas regime, not about the solvers themselves. If they disagree on raw too, the disagreement is intrinsic to the priors/regularizers.

The raw HGP FITS is produced with the same length scales / hierarchy as the head-residual run, only with `--offset-kind raw`:

```bash
python -m models.astrometry2.fit_hierarchical_gp_concordance \
    --anchors models/checkpoints/latent_position_v10_no_psf/anchors_centernet_v10.npz \
    --output  models/checkpoints/latent_position_v10_no_psf/concordance_hgp_centernet_v10_raw_dedup.fits \
    --offset-kind raw --bands u,g,r,i,z,y,nisp_Y,nisp_J,nisp_H \
    --length-scales 45,120,300,900 --max-centers-per-scale 120 \
    --prior-common-mas 25.0 --prior-group-mas 12.0 --prior-band-mas 6.0 \
    --robust-iters 3 --huber-k 3.0 --dstep-arcsec 5.0 \
    --holdout-mode spatial --save-components --write-coverage --seed 42
```

In [ ]:
HGP_RAW_FITS = CKPT / 'concordance_hgp_centernet_v10_raw_dedup.fits'
HGP_RAW_JSON = HGP_RAW_FITS.with_suffix('.fits.json')


def have_raw_hgp():
    if not HGP_RAW_FITS.exists():
        print(f"Raw-anchor HGP FITS not found at {HGP_RAW_FITS.relative_to(ROOT)}.")
        print("Run the CLI command from the markdown above to produce it (~8 min on this anchor cache),")
        print("then re-execute this section.")
        return False
    return True


def load_resampled_band_raw(band):
    """Sample HGP_RAW and PINN_RAW on the HGP_RAW grid for one band."""
    with fits.open(HGP_RAW_FITS, memmap=True) as hgp, \
         fits.open(PINN_RAW_FITS, memmap=True) as pinn:
        h_hdu = hgp[f'{band}.DRA']
        ra, dec = grid_world_from_header(h_hdu.header)
        return {
            'ra': ra, 'dec': dec,
            'hgp_dra': np.asarray(hgp[f'{band}.DRA'].data, dtype=np.float64),
            'hgp_dde': np.asarray(hgp[f'{band}.DDE'].data, dtype=np.float64),
            'hgp_dra_std': np.asarray(hgp[f'{band}.DRA_STD'].data, dtype=np.float64),
            'hgp_dde_std': np.asarray(hgp[f'{band}.DDE_STD'].data, dtype=np.float64),
            'hgp_coverage': np.asarray(hgp['COVERAGE'].data, dtype=np.float64) if 'COVERAGE' in hgp else np.ones_like(ra),
            'pinn_dra': sample_image_at_world(pinn[f'{band}.DRA'], ra, dec),
            'pinn_dde': sample_image_at_world(pinn[f'{band}.DDE'], ra, dec),
            'pinn_coverage': sample_image_at_world(pinn['COVERAGE'], ra, dec) if 'COVERAGE' in pinn else np.ones_like(ra),
        }


def finite_mask_raw(d):
    finite = np.isfinite(d['hgp_dra']) & np.isfinite(d['hgp_dde'])
    finite &= np.isfinite(d['pinn_dra']) & np.isfinite(d['pinn_dde'])
    finite &= np.isfinite(d['hgp_coverage']) & np.isfinite(d['pinn_coverage'])
    return finite


def support_mask_raw(d, max_anchor_dist_arcsec=MAX_NEAREST_ANCHOR_ARCSEC, max_hgp_std_mas=MAX_HGP_STD_RADIAL_MAS):
    mask = finite_mask_raw(d)
    mask &= d['hgp_coverage'] <= max_anchor_dist_arcsec
    mask &= d['pinn_coverage'] <= max_anchor_dist_arcsec
    if max_hgp_std_mas is not None:
        mask &= amp_mas(d['hgp_dra_std'], d['hgp_dde_std']) <= max_hgp_std_mas
    return mask


if have_raw_hgp():
    rows_raw = []
    for band in BANDS:
        d = load_resampled_band_raw(band)
        m_full = finite_mask_raw(d)
        m = support_mask_raw(d)
        diff_dra = d['hgp_dra'] - d['pinn_dra']
        diff_dde = d['hgp_dde'] - d['pinn_dde']
        std_radial = amp_mas(d['hgp_dra_std'], d['hgp_dde_std'])
        rows_raw.append({
            'band': BAND_LABEL[band],
            'n_full': int(m_full.sum()),
            'n_support': int(m.sum()),
            'support_frac': float(m.sum() / max(m_full.sum(), 1)),
            'hgp_raw_rms_full_mas': rms_amp_mas(d['hgp_dra'], d['hgp_dde'], m_full),
            'hgp_raw_median_support_mas': median_amp_mas(d['hgp_dra'], d['hgp_dde'], m),
            'hgp_raw_rms_support_mas': rms_amp_mas(d['hgp_dra'], d['hgp_dde'], m),
            'pinn_raw_rms_support_mas': rms_amp_mas(d['pinn_dra'], d['pinn_dde'], m),
            'hgp_minus_pinn_raw_rms_support_mas': rms_amp_mas(diff_dra, diff_dde, m),
            'hgp_raw_median_std_support_mas': float(np.nanmedian(std_radial[m])),
        })
    compare_raw_df = pd.DataFrame(rows_raw)
    display(compare_raw_df.round(3))

In [ ]:
if have_raw_hgp():
    PLOT_BAND_RAW = 'I'
    d = load_resampled_band_raw(PLOT_BAND_RAW)
    m_full = finite_mask_raw(d)
    m = support_mask_raw(d)

    maps_raw = {
        'HGP raw |F| (full)':              (amp_mas(d['hgp_dra'], d['hgp_dde']), m_full, 'mas'),
        'PINN raw |F| (support)':          (amp_mas(d['pinn_dra'], d['pinn_dde']), m, 'mas'),
        '|HGP raw - PINN raw| (support)':  (amp_mas(d['hgp_dra'] - d['pinn_dra'], d['hgp_dde'] - d['pinn_dde']), m, 'mas'),
        'HGP raw posterior std (full)':    (amp_mas(d['hgp_dra_std'], d['hgp_dde_std']), m_full, 'mas'),
        'Nearest anchor dist':             (d['hgp_coverage'], m_full, 'arcsec'),
        'Support mask':                    (m.astype(float), m_full, 'mask'),
    }

    fig, axes = plt.subplots(2, 3, figsize=(10.5, 8.4), constrained_layout=True)
    for ax, (title, (image, mask, unit)) in zip(axes.ravel(), maps_raw.items()):
        show = np.asarray(image, dtype=np.float64).copy()
        show[~mask] = np.nan
        if unit == 'arcsec':
            vmin, vmax, cmap = 0, np.nanpercentile(show, 99), 'viridis'
            label = 'nearest anchor [arcsec]'
        elif unit == 'mask':
            vmin, vmax, cmap = 0, 1, 'gray_r'
            label = 'support'
        else:
            vmin, vmax, cmap = 0, np.nanpercentile(show, 98), 'magma'
            label = 'mas'
        im = ax.imshow(show, origin='lower', cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02, label=label)
    fig.suptitle(f'{BAND_LABEL[PLOT_BAND_RAW]}: HGP raw vs PINN raw on the HGP-raw grid', y=1.02)

In [ ]:
if have_raw_hgp():
    anchor_raw_rows = []
    with fits.open(HGP_RAW_FITS, memmap=True) as hgp_raw, \
         fits.open(PINN_RAW_FITS, memmap=True) as pinn_raw:
        for band in BANDS:
            key = BAND_KEY[band]
            ra = anchors[f'{key}_ra']
            dec = anchors[f'{key}_dec']
            off = anchors[f'{key}_raw'].astype(np.float64)
            hgp_pred = sample_field_pair(hgp_raw, band, ra, dec)
            pinn_pred = sample_field_pair(pinn_raw, band, ra, dec)
            hgp_std = sample_field_pair(hgp_raw, band, ra, dec, suffix='_STD')
            hgp_dist = sample_image_at_world(hgp_raw['COVERAGE'], ra, dec)
            pinn_dist = sample_image_at_world(pinn_raw['COVERAGE'], ra, dec)
            finite = (
                np.isfinite(off).all(axis=1)
                & np.isfinite(hgp_pred).all(axis=1)
                & np.isfinite(pinn_pred).all(axis=1)
                & np.isfinite(hgp_std).all(axis=1)
                & np.isfinite(hgp_dist) & np.isfinite(pinn_dist)
            )
            finite &= amp_mas(off[:, 0], off[:, 1]) < 200.0
            support = finite & (hgp_dist <= MAX_NEAREST_ANCHOR_ARCSEC) & (pinn_dist <= MAX_NEAREST_ANCHOR_ARCSEC)
            support &= amp_mas(hgp_std[:, 0], hgp_std[:, 1]) <= MAX_HGP_STD_RADIAL_MAS
            for name, mask in [('finite', finite), ('support', support)]:
                if not mask.any():
                    continue
                zero = amp_mas(off[mask, 0], off[mask, 1])
                after_hgp = amp_mas(off[mask, 0] - hgp_pred[mask, 0], off[mask, 1] - hgp_pred[mask, 1])
                after_pinn = amp_mas(off[mask, 0] - pinn_pred[mask, 0], off[mask, 1] - pinn_pred[mask, 1])
                anchor_raw_rows.append({
                    'band': BAND_LABEL[band],
                    'mask': name,
                    'n': int(mask.sum()),
                    'zero_field_med_mas': float(np.nanmedian(zero)),
                    'hgp_after_med_mas': float(np.nanmedian(after_hgp)),
                    'pinn_after_med_mas': float(np.nanmedian(after_pinn)),
                    'hgp_delta_med_mas': float(np.nanmedian(zero) - np.nanmedian(after_hgp)),
                    'pinn_delta_med_mas': float(np.nanmedian(zero) - np.nanmedian(after_pinn)),
                    'hgp_field_med_mas': median_amp_mas(hgp_pred[:, 0], hgp_pred[:, 1], mask),
                    'pinn_field_med_mas': median_amp_mas(pinn_pred[:, 0], pinn_pred[:, 1], mask),
                    'hgp_std_med_mas': median_amp_mas(hgp_std[:, 0], hgp_std[:, 1], mask),
                })

    anchor_raw_df = pd.DataFrame(anchor_raw_rows)
    display(anchor_raw_df.round(3))

    weighted_raw = []
    for mask_name, group in anchor_raw_df.groupby('mask'):
        weights = group['n'].to_numpy()
        weighted_raw.append({
            'mask': mask_name,
            'n_total': int(weights.sum()),
            'hgp_delta_source_weighted_mas': float(np.average(group['hgp_delta_med_mas'], weights=weights)),
            'pinn_delta_source_weighted_mas': float(np.average(group['pinn_delta_med_mas'], weights=weights)),
        })
    display(pd.DataFrame(weighted_raw).round(4))

### How to read this section

- **Same field, two solvers**: in section 5 the head-residual baseline is ~9 mas per source and the smooth field is ~1 mas. On raw, the per-source baseline is ~8-9 mas (Rubin) / ~7-8 mas (NISP) too — but the *coherent* component is much larger, so subtracting either field should bite. If `hgp_delta_med_mas` and `pinn_delta_med_mas` are now both clearly positive (e.g. tenths of a mas to ~1 mas), there is a real signal here that the head-residual test was below; if they are still essentially tied with zero, there is no smooth concordance to recover even on raw.
- **`hgp_minus_pinn_raw_rms_support_mas`** is the model-disagreement scale on raw. Compare it to each solver's individual support RMS. Agreement is method-independence; disagreement at >~1-2 mas means the priors are doing real work even on raw, so the head-resid disagreement above is partly inherited.
- **HGP raw posterior std** should be smaller than the supported HGP raw RMS (signal > uncertainty). If it is comparable, the field is not detected over the HGP noise model.
- **NISP vs Rubin**: the PINN raw fit shows NISP at ~7 mas and Rubin (g/r/i/z) at ~5 mas. If HGP roughly tracks that ordering, the Rubin/NISP common+group structure in section 4 is consistent. If HGP collapses everything to a single common field with similar amplitude across instruments, that points back to the prior absorbing real per-instrument structure.
- **Compared to the head-residual test (section 5)**: a clear `pinn_delta` and `hgp_delta` here, alongside near-zero deltas there, is the cleanest way to say "the head ate the smooth field"; deltas that are similarly small in both raw and head-residual would say "no smooth field is recoverable at all in this footprint."

## 7. Tighter-prior HGP raw refit

The Section 6 HGP raw map traced per-tile noise instead of a smooth degree-scale field, with `|HGP - PINN|` of order the PINN signal itself. This section repeats the HGP raw fit with two changes designed to suppress that failure mode:

1. **Drop the 45 arcsec length scale**: keep only 120/300/900. Per-tile structure cannot be fit by basis functions whose minimum scale is larger than a typical tile span.
2. **Tighten the hierarchical priors**: `--prior-common-mas 8` (was 25), `--prior-group-mas 6` (was 12), `--prior-band-mas 4` (was 6). The looser values let the common component absorb noise; tightening forces HGP to spend amplitude only on structure shared across instruments at degree scale.

```bash
python -m models.astrometry2.fit_hierarchical_gp_concordance \
    --anchors models/checkpoints/latent_position_v10_no_psf/anchors_centernet_v10.npz \
    --output  models/checkpoints/latent_position_v10_no_psf/concordance_hgp_centernet_v10_raw_tight_dedup.fits \
    --offset-kind raw --bands u,g,r,i,z,y,nisp_Y,nisp_J,nisp_H \
    --length-scales 120,300,900 --max-centers-per-scale 120 \
    --prior-common-mas 8.0 --prior-group-mas 6.0 --prior-band-mas 4.0 \
    --robust-iters 3 --huber-k 3.0 --dstep-arcsec 5.0 \
    --holdout-mode spatial --save-components --write-coverage --seed 42
```

Success criterion: `|HGP_tight - PINN|` RMS drops well below the PINN signal, and the HGP map looks smooth (no tile criss-cross). Failure means HGP's hierarchical-Gaussian formulation cannot mimic PINN's curl-free / Laplacian / band-consistency priors no matter how it is tuned, and the trustworthy raw cross-checks remain PINN, NN, and the sklearn GP from Notebook 07.

In [ ]:
HGP_RAW_TIGHT_FITS = CKPT / 'concordance_hgp_centernet_v10_raw_tight_dedup.fits'


def have_raw_hgp_tight():
    if not HGP_RAW_TIGHT_FITS.exists():
        print(f"Tight raw HGP FITS not found at {HGP_RAW_TIGHT_FITS.relative_to(ROOT)}.")
        return False
    return True


def load_resampled_band_raw_tight(band):
    with fits.open(HGP_RAW_TIGHT_FITS, memmap=True) as hgp, \
         fits.open(PINN_RAW_FITS, memmap=True) as pinn, \
         fits.open(HGP_RAW_FITS, memmap=True) as hgp_old:
        h_hdu = hgp[f'{band}.DRA']
        ra, dec = grid_world_from_header(h_hdu.header)
        return {
            'ra': ra, 'dec': dec,
            'hgp_dra': np.asarray(hgp[f'{band}.DRA'].data, dtype=np.float64),
            'hgp_dde': np.asarray(hgp[f'{band}.DDE'].data, dtype=np.float64),
            'hgp_dra_std': np.asarray(hgp[f'{band}.DRA_STD'].data, dtype=np.float64),
            'hgp_dde_std': np.asarray(hgp[f'{band}.DDE_STD'].data, dtype=np.float64),
            'hgp_coverage': np.asarray(hgp['COVERAGE'].data, dtype=np.float64) if 'COVERAGE' in hgp else np.ones_like(ra),
            'pinn_dra': sample_image_at_world(pinn[f'{band}.DRA'], ra, dec),
            'pinn_dde': sample_image_at_world(pinn[f'{band}.DDE'], ra, dec),
            'pinn_coverage': sample_image_at_world(pinn['COVERAGE'], ra, dec) if 'COVERAGE' in pinn else np.ones_like(ra),
            'hgp_old_dra': sample_image_at_world(hgp_old[f'{band}.DRA'], ra, dec),
            'hgp_old_dde': sample_image_at_world(hgp_old[f'{band}.DDE'], ra, dec),
        }


def support_mask_tight(d, max_anchor_dist=MAX_NEAREST_ANCHOR_ARCSEC, max_hgp_std_mas=MAX_HGP_STD_RADIAL_MAS):
    mask = np.isfinite(d['hgp_dra']) & np.isfinite(d['hgp_dde'])
    mask &= np.isfinite(d['pinn_dra']) & np.isfinite(d['pinn_dde'])
    mask &= np.isfinite(d['hgp_old_dra']) & np.isfinite(d['hgp_old_dde'])
    mask &= np.isfinite(d['hgp_coverage']) & np.isfinite(d['pinn_coverage'])
    mask &= d['hgp_coverage'] <= max_anchor_dist
    mask &= d['pinn_coverage'] <= max_anchor_dist
    if max_hgp_std_mas is not None:
        mask &= amp_mas(d['hgp_dra_std'], d['hgp_dde_std']) <= max_hgp_std_mas
    return mask


if have_raw_hgp_tight():
    rows_tight = []
    for band in BANDS:
        d = load_resampled_band_raw_tight(band)
        m = support_mask_tight(d)
        rows_tight.append({
            'band': BAND_LABEL[band],
            'n_support': int(m.sum()),
            'hgp_tight_rms_mas': rms_amp_mas(d['hgp_dra'], d['hgp_dde'], m),
            'hgp_orig_rms_mas': rms_amp_mas(d['hgp_old_dra'], d['hgp_old_dde'], m),
            'pinn_rms_mas': rms_amp_mas(d['pinn_dra'], d['pinn_dde'], m),
            'tight_minus_pinn_rms_mas': rms_amp_mas(d['hgp_dra'] - d['pinn_dra'], d['hgp_dde'] - d['pinn_dde'], m),
            'orig_minus_pinn_rms_mas': rms_amp_mas(d['hgp_old_dra'] - d['pinn_dra'], d['hgp_old_dde'] - d['pinn_dde'], m),
            'tight_minus_orig_rms_mas': rms_amp_mas(d['hgp_dra'] - d['hgp_old_dra'], d['hgp_dde'] - d['hgp_old_dde'], m),
            'hgp_tight_std_mas': float(np.nanmedian(amp_mas(d['hgp_dra_std'], d['hgp_dde_std'])[m])),
        })
    compare_tight_df = pd.DataFrame(rows_tight)
    display(compare_tight_df.round(3))

In [ ]:
if have_raw_hgp_tight():
    PLOT_BAND_TIGHT = 'I'
    d = load_resampled_band_raw_tight(PLOT_BAND_TIGHT)
    m = support_mask_tight(d)

    panels = {
        'HGP tight |F|':       (amp_mas(d['hgp_dra'], d['hgp_dde']),                 m, 'mas'),
        'HGP original |F|':    (amp_mas(d['hgp_old_dra'], d['hgp_old_dde']),         m, 'mas'),
        'PINN raw |F|':        (amp_mas(d['pinn_dra'], d['pinn_dde']),               m, 'mas'),
        '|HGP tight - PINN|':  (amp_mas(d['hgp_dra'] - d['pinn_dra'], d['hgp_dde'] - d['pinn_dde']),     m, 'mas'),
        '|HGP orig - PINN|':   (amp_mas(d['hgp_old_dra'] - d['pinn_dra'], d['hgp_old_dde'] - d['pinn_dde']), m, 'mas'),
        'HGP tight std':       (amp_mas(d['hgp_dra_std'], d['hgp_dde_std']),         m, 'mas'),
    }
    fig, axes = plt.subplots(2, 3, figsize=(10.5, 8.4), constrained_layout=True)
    for ax, (title, (image, mask, unit)) in zip(axes.ravel(), panels.items()):
        show = np.asarray(image, dtype=np.float64).copy()
        show[~mask] = np.nan
        vmax = np.nanpercentile(show, 98)
        im = ax.imshow(show, origin='lower', cmap='magma', vmin=0, vmax=vmax)
        ax.set_title(title)
        ax.set_xticks([]); ax.set_yticks([])
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02, label='mas')
    fig.suptitle(f'{BAND_LABEL[PLOT_BAND_TIGHT]}: tight-prior HGP vs original HGP vs PINN, raw anchors', y=1.02)

In [ ]:
if have_raw_hgp_tight():
    anchor_tight_rows = []
    with fits.open(HGP_RAW_TIGHT_FITS, memmap=True) as hgp_tight, \
         fits.open(PINN_RAW_FITS, memmap=True) as pinn_raw:
        for band in BANDS:
            key = BAND_KEY[band]
            ra = anchors[f'{key}_ra']
            dec = anchors[f'{key}_dec']
            off = anchors[f'{key}_raw'].astype(np.float64)
            hgp_pred = sample_field_pair(hgp_tight, band, ra, dec)
            pinn_pred = sample_field_pair(pinn_raw, band, ra, dec)
            hgp_std = sample_field_pair(hgp_tight, band, ra, dec, suffix='_STD')
            hgp_dist = sample_image_at_world(hgp_tight['COVERAGE'], ra, dec)
            pinn_dist = sample_image_at_world(pinn_raw['COVERAGE'], ra, dec)
            finite = (
                np.isfinite(off).all(axis=1)
                & np.isfinite(hgp_pred).all(axis=1)
                & np.isfinite(pinn_pred).all(axis=1)
                & np.isfinite(hgp_std).all(axis=1)
                & np.isfinite(hgp_dist) & np.isfinite(pinn_dist)
            )
            finite &= amp_mas(off[:, 0], off[:, 1]) < 200.0
            support = finite & (hgp_dist <= MAX_NEAREST_ANCHOR_ARCSEC) & (pinn_dist <= MAX_NEAREST_ANCHOR_ARCSEC)
            support &= amp_mas(hgp_std[:, 0], hgp_std[:, 1]) <= MAX_HGP_STD_RADIAL_MAS
            for name, mask in [('finite', finite), ('support', support)]:
                if not mask.any():
                    continue
                zero = amp_mas(off[mask, 0], off[mask, 1])
                after_hgp = amp_mas(off[mask, 0] - hgp_pred[mask, 0], off[mask, 1] - hgp_pred[mask, 1])
                after_pinn = amp_mas(off[mask, 0] - pinn_pred[mask, 0], off[mask, 1] - pinn_pred[mask, 1])
                anchor_tight_rows.append({
                    'band': BAND_LABEL[band], 'mask': name, 'n': int(mask.sum()),
                    'zero_field_med_mas': float(np.nanmedian(zero)),
                    'hgp_tight_after_med_mas': float(np.nanmedian(after_hgp)),
                    'pinn_after_med_mas': float(np.nanmedian(after_pinn)),
                    'hgp_tight_delta_med_mas': float(np.nanmedian(zero) - np.nanmedian(after_hgp)),
                    'pinn_delta_med_mas': float(np.nanmedian(zero) - np.nanmedian(after_pinn)),
                })

    anchor_tight_df = pd.DataFrame(anchor_tight_rows)
    display(anchor_tight_df.round(3))

    weighted_tight = []
    for mask_name, group in anchor_tight_df.groupby('mask'):
        weights = group['n'].to_numpy()
        weighted_tight.append({
            'mask': mask_name, 'n_total': int(weights.sum()),
            'hgp_tight_delta_source_weighted_mas': float(np.average(group['hgp_tight_delta_med_mas'], weights=weights)),
            'pinn_delta_source_weighted_mas': float(np.average(group['pinn_delta_med_mas'], weights=weights)),
        })
    display(pd.DataFrame(weighted_tight).round(4))

### 7b. Super-tight HGP raw refit

The Section 7 tight HGP map still showed visible tile criss-cross. Push priors further: drop the 120 arcsec scale (keep only 300/900) and tighten the hierarchy aggressively (`common 4 mas, group 2 mas, band 1 mas`). At this setting HGP can only fit degree-scale structure shared across instruments, so if it still produces a smooth field that tracks PINN we have a working cross-check; if criss-cross persists, the HGP basis-function GP cannot match PINN's curl-free / Laplacian / band-consistency physics priors at any setting and HGP is structurally unable to serve as a like-for-like cross-validator on this geometry.

```bash
python -m models.astrometry2.fit_hierarchical_gp_concordance \
    --anchors models/checkpoints/latent_position_v10_no_psf/anchors_centernet_v10.npz \
    --output  models/checkpoints/latent_position_v10_no_psf/concordance_hgp_centernet_v10_raw_supertight_dedup.fits \
    --offset-kind raw --bands u,g,r,i,z,y,nisp_Y,nisp_J,nisp_H \
    --length-scales 300,900 --max-centers-per-scale 120 \
    --prior-common-mas 4.0 --prior-group-mas 2.0 --prior-band-mas 1.0 \
    --robust-iters 3 --huber-k 3.0 --dstep-arcsec 5.0 \
    --holdout-mode spatial --save-components --write-coverage --seed 42
```


In [ ]:
HGP_RAW_SUPERTIGHT_FITS = CKPT / 'concordance_hgp_centernet_v10_raw_supertight_dedup.fits'


def have_raw_hgp_supertight():
    if not HGP_RAW_SUPERTIGHT_FITS.exists():
        print(f"Super-tight raw HGP FITS not found at {HGP_RAW_SUPERTIGHT_FITS.relative_to(ROOT)}.")
        return False
    return True


def load_resampled_band_supertight(band):
    with fits.open(HGP_RAW_SUPERTIGHT_FITS, memmap=True) as hgp_st, \
         fits.open(HGP_RAW_TIGHT_FITS, memmap=True) as hgp_t, \
         fits.open(PINN_RAW_FITS, memmap=True) as pinn:
        h_hdu = hgp_st[f'{band}.DRA']
        ra, dec = grid_world_from_header(h_hdu.header)
        return {
            'ra': ra, 'dec': dec,
            'hgp_dra': np.asarray(hgp_st[f'{band}.DRA'].data, dtype=np.float64),
            'hgp_dde': np.asarray(hgp_st[f'{band}.DDE'].data, dtype=np.float64),
            'hgp_dra_std': np.asarray(hgp_st[f'{band}.DRA_STD'].data, dtype=np.float64),
            'hgp_dde_std': np.asarray(hgp_st[f'{band}.DDE_STD'].data, dtype=np.float64),
            'hgp_coverage': np.asarray(hgp_st['COVERAGE'].data, dtype=np.float64) if 'COVERAGE' in hgp_st else np.ones_like(ra),
            'hgp_tight_dra': sample_image_at_world(hgp_t[f'{band}.DRA'], ra, dec),
            'hgp_tight_dde': sample_image_at_world(hgp_t[f'{band}.DDE'], ra, dec),
            'pinn_dra': sample_image_at_world(pinn[f'{band}.DRA'], ra, dec),
            'pinn_dde': sample_image_at_world(pinn[f'{band}.DDE'], ra, dec),
            'pinn_coverage': sample_image_at_world(pinn['COVERAGE'], ra, dec) if 'COVERAGE' in pinn else np.ones_like(ra),
        }


def support_mask_supertight(d, max_anchor_dist=MAX_NEAREST_ANCHOR_ARCSEC, max_hgp_std_mas=MAX_HGP_STD_RADIAL_MAS):
    mask = np.isfinite(d['hgp_dra']) & np.isfinite(d['hgp_dde'])
    mask &= np.isfinite(d['pinn_dra']) & np.isfinite(d['pinn_dde'])
    mask &= np.isfinite(d['hgp_tight_dra']) & np.isfinite(d['hgp_tight_dde'])
    mask &= np.isfinite(d['hgp_coverage']) & np.isfinite(d['pinn_coverage'])
    mask &= d['hgp_coverage'] <= max_anchor_dist
    mask &= d['pinn_coverage'] <= max_anchor_dist
    if max_hgp_std_mas is not None:
        mask &= amp_mas(d['hgp_dra_std'], d['hgp_dde_std']) <= max_hgp_std_mas
    return mask


if have_raw_hgp_supertight():
    rows_st = []
    for band in BANDS:
        d = load_resampled_band_supertight(band)
        m = support_mask_supertight(d)
        rows_st.append({
            'band': BAND_LABEL[band],
            'n_support': int(m.sum()),
            'hgp_supertight_rms_mas': rms_amp_mas(d['hgp_dra'], d['hgp_dde'], m),
            'hgp_tight_rms_mas': rms_amp_mas(d['hgp_tight_dra'], d['hgp_tight_dde'], m),
            'pinn_rms_mas': rms_amp_mas(d['pinn_dra'], d['pinn_dde'], m),
            'supertight_minus_pinn_rms_mas': rms_amp_mas(d['hgp_dra'] - d['pinn_dra'], d['hgp_dde'] - d['pinn_dde'], m),
            'supertight_minus_tight_rms_mas': rms_amp_mas(d['hgp_dra'] - d['hgp_tight_dra'], d['hgp_dde'] - d['hgp_tight_dde'], m),
            'hgp_supertight_std_mas': float(np.nanmedian(amp_mas(d['hgp_dra_std'], d['hgp_dde_std'])[m])),
        })
    compare_st_df = pd.DataFrame(rows_st)
    display(compare_st_df.round(3))


In [ ]:
if have_raw_hgp_supertight():
    PLOT_BAND_ST = 'I'
    d = load_resampled_band_supertight(PLOT_BAND_ST)
    m = support_mask_supertight(d)

    panels = {
        'HGP super-tight |F|': amp_mas(d['hgp_dra'], d['hgp_dde']),
        'PINN raw |F|':        amp_mas(d['pinn_dra'], d['pinn_dde']),
    }
    vmax = float(np.nanpercentile(
        np.concatenate([np.where(m, v, np.nan).ravel() for v in panels.values()]),
        98,
    ))
    fig, axes = plt.subplots(1, 2, figsize=(8, 5.6), constrained_layout=True)
    for ax, (title, image) in zip(axes, panels.items()):
        show = np.asarray(image, dtype=np.float64).copy()
        show[~m] = np.nan
        im = ax.imshow(show, origin='lower', cmap='magma', vmin=0, vmax=vmax)
        ax.set_title(title)
        ax.set_xticks([]); ax.set_yticks([])
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02, label='mas')
    fig.suptitle(f'{BAND_LABEL[PLOT_BAND_ST]}: super-tight HGP vs PINN, raw anchors', y=1.03)


In [ ]:
if have_raw_hgp_supertight():
    anchor_st_rows = []
    with fits.open(HGP_RAW_SUPERTIGHT_FITS, memmap=True) as hgp_st, \
         fits.open(PINN_RAW_FITS, memmap=True) as pinn_raw:
        for band in BANDS:
            key = BAND_KEY[band]
            ra = anchors[f'{key}_ra']
            dec = anchors[f'{key}_dec']
            off = anchors[f'{key}_raw'].astype(np.float64)
            hgp_pred = sample_field_pair(hgp_st, band, ra, dec)
            pinn_pred = sample_field_pair(pinn_raw, band, ra, dec)
            hgp_std = sample_field_pair(hgp_st, band, ra, dec, suffix='_STD')
            hgp_dist = sample_image_at_world(hgp_st['COVERAGE'], ra, dec)
            pinn_dist = sample_image_at_world(pinn_raw['COVERAGE'], ra, dec)
            finite = (
                np.isfinite(off).all(axis=1)
                & np.isfinite(hgp_pred).all(axis=1)
                & np.isfinite(pinn_pred).all(axis=1)
                & np.isfinite(hgp_std).all(axis=1)
                & np.isfinite(hgp_dist) & np.isfinite(pinn_dist)
            )
            finite &= amp_mas(off[:, 0], off[:, 1]) < 200.0
            support = finite & (hgp_dist <= MAX_NEAREST_ANCHOR_ARCSEC) & (pinn_dist <= MAX_NEAREST_ANCHOR_ARCSEC)
            support &= amp_mas(hgp_std[:, 0], hgp_std[:, 1]) <= MAX_HGP_STD_RADIAL_MAS
            for name, mask in [('finite', finite), ('support', support)]:
                if not mask.any():
                    continue
                zero = amp_mas(off[mask, 0], off[mask, 1])
                after_hgp = amp_mas(off[mask, 0] - hgp_pred[mask, 0], off[mask, 1] - hgp_pred[mask, 1])
                after_pinn = amp_mas(off[mask, 0] - pinn_pred[mask, 0], off[mask, 1] - pinn_pred[mask, 1])
                anchor_st_rows.append({
                    'band': BAND_LABEL[band], 'mask': name, 'n': int(mask.sum()),
                    'zero_field_med_mas': float(np.nanmedian(zero)),
                    'hgp_st_after_med_mas': float(np.nanmedian(after_hgp)),
                    'pinn_after_med_mas': float(np.nanmedian(after_pinn)),
                    'hgp_st_delta_med_mas': float(np.nanmedian(zero) - np.nanmedian(after_hgp)),
                    'pinn_delta_med_mas': float(np.nanmedian(zero) - np.nanmedian(after_pinn)),
                })

    anchor_st_df = pd.DataFrame(anchor_st_rows)
    display(anchor_st_df.round(3))

    weighted_st = []
    for mask_name, group in anchor_st_df.groupby('mask'):
        weights = group['n'].to_numpy()
        weighted_st.append({
            'mask': mask_name, 'n_total': int(weights.sum()),
            'hgp_st_delta_source_weighted_mas': float(np.average(group['hgp_st_delta_med_mas'], weights=weights)),
            'pinn_delta_source_weighted_mas': float(np.average(group['pinn_delta_med_mas'], weights=weights)),
        })
    display(pd.DataFrame(weighted_st).round(4))


### 7c. Where does the criss-cross come from? Anchor density on the HGP grid

The super-tight HGP map still shows a faint cellular network. The simplest first diagnostic is to bin the actual anchor positions onto the same 5 arcsec mesh and overlay the high-density contours on the HGP field. If the bright nodes coincide with high-density anchor clumps (most likely sources detected in multiple overlapping tiles, where each tile's WCS solution differs by a few mas), the structure is data-driven and would be present for any GP at this scale; PINN happens to suppress it because the curl-free / Laplacian priors penalize the rotational pattern that anchor clumps tend to induce.

If instead the bright nodes have no relation to anchor density, the cells are a basis-lattice artifact (300 arcsec RBFs evaluating to ridges between centers) and a different mitigation is needed (more centers per scale, or a continuous GP).


In [ ]:
if have_raw_hgp_supertight():
    PLOT_BAND_DIAG = 'I'
    key = BAND_KEY[PLOT_BAND_DIAG]
    BIN_BLOCK = 12  # 12 pixels x 5 arcsec/pixel = 60 arcsec bins

    with fits.open(HGP_RAW_SUPERTIGHT_FITS, memmap=True) as h:
        hdu = h[f'{PLOT_BAND_DIAG}.DRA']
        wcs = WCS(hdu.header)
        ny, nx = hdu.data.shape
        hgp_amp = amp_mas(
            np.asarray(hdu.data, dtype=np.float64),
            np.asarray(h[f'{PLOT_BAND_DIAG}.DDE'].data, dtype=np.float64),
        )
        cov = np.asarray(h['COVERAGE'].data, dtype=np.float64) if 'COVERAGE' in h else np.ones((ny, nx))
        std_amp = amp_mas(
            np.asarray(h[f'{PLOT_BAND_DIAG}.DRA_STD'].data, dtype=np.float64),
            np.asarray(h[f'{PLOT_BAND_DIAG}.DDE_STD'].data, dtype=np.float64),
        )

    m = (cov <= MAX_NEAREST_ANCHOR_ARCSEC) & (std_amp <= MAX_HGP_STD_RADIAL_MAS) & np.isfinite(hgp_amp)

    # Bin anchors into BIN_BLOCK x BIN_BLOCK pixel cells (60 arcsec at 5"/pix)
    anc_ra = anchors[f'{key}_ra']
    anc_dec = anchors[f'{key}_dec']
    finite = np.isfinite(anc_ra) & np.isfinite(anc_dec)
    x, y = wcs.wcs_world2pix(anc_ra[finite], anc_dec[finite], 0)
    in_grid = (x >= 0) & (x < nx) & (y >= 0) & (y < ny)
    n_anchors_in_grid = int(in_grid.sum())

    coarse_ny = ny // BIN_BLOCK
    coarse_nx = nx // BIN_BLOCK
    density_coarse, _, _ = np.histogram2d(
        y[in_grid], x[in_grid],
        bins=(coarse_ny, coarse_nx),
        range=[[0, coarse_ny * BIN_BLOCK], [0, coarse_nx * BIN_BLOCK]],
    )
    # Upsample density to the full HGP grid and smooth for nicer contours
    from scipy.ndimage import gaussian_filter
    density_hi = np.repeat(np.repeat(density_coarse, BIN_BLOCK, axis=0), BIN_BLOCK, axis=1)
    pad_y = ny - density_hi.shape[0]
    pad_x = nx - density_hi.shape[1]
    if pad_y > 0 or pad_x > 0:
        density_hi = np.pad(density_hi, ((0, pad_y), (0, pad_x)), mode='edge')
    density_hi = gaussian_filter(density_hi, sigma=BIN_BLOCK / 2.0)

    hgp_show = hgp_amp.copy(); hgp_show[~m] = np.nan
    vmax_hgp = float(np.nanpercentile(hgp_show, 98))

    # Mean density per arcmin^2 (60 arcsec bin = 1 arcmin^2 exactly)
    den_show_coarse = density_coarse.copy()
    vmax_den = float(np.nanpercentile(den_show_coarse[den_show_coarse > 0], 98)) if (den_show_coarse > 0).any() else 1.0

    fig, axes = plt.subplots(1, 3, figsize=(12, 5.4), constrained_layout=True)

    im = axes[0].imshow(hgp_show, origin='lower', cmap='magma', vmin=0, vmax=vmax_hgp)
    axes[0].set_title(f'HGP super-tight |F| ({BAND_LABEL[PLOT_BAND_DIAG]})')
    axes[0].set_xticks([]); axes[0].set_yticks([])
    fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.02, label='mas')

    im = axes[1].imshow(
        den_show_coarse, origin='lower', cmap='viridis', vmin=0, vmax=vmax_den,
        extent=(0, coarse_nx * BIN_BLOCK, 0, coarse_ny * BIN_BLOCK), aspect='equal',
    )
    axes[1].set_xlim(0, nx); axes[1].set_ylim(0, ny)
    axes[1].set_title(f'{BAND_LABEL[PLOT_BAND_DIAG]} anchor density per arcmin^2\nN={n_anchors_in_grid} anchors')
    axes[1].set_xticks([]); axes[1].set_yticks([])
    fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.02, label='anchors / arcmin^2')

    im = axes[2].imshow(hgp_show, origin='lower', cmap='magma', vmin=0, vmax=vmax_hgp)
    if vmax_den > 0:
        levels = np.linspace(0.5 * vmax_den, vmax_den, 4)
        axes[2].contour(
            density_hi[:ny, :nx], levels=levels, colors='cyan', linewidths=1.4, alpha=0.95,
        )
    axes[2].set_title('HGP with arcmin-scale density contours')
    axes[2].set_xticks([]); axes[2].set_yticks([])
    fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.02, label='mas')

    fig.suptitle(f'Section 7c: arcmin-binned anchor density vs HGP super-tight ({BAND_LABEL[PLOT_BAND_DIAG]})', y=1.03)


In [ ]:
if have_raw_hgp_supertight():
    PLOT_BAND_ST_SCATTER = 'I'
    d = load_resampled_band_supertight(PLOT_BAND_ST_SCATTER)
    m = support_mask_supertight(d)

    rng = np.random.default_rng(42)
    idx = np.flatnonzero(m.ravel())
    idx = rng.choice(idx, size=min(8000, idx.size), replace=False)

    hgp_dra = d['hgp_dra'].ravel()[idx] * 1000
    hgp_dde = d['hgp_dde'].ravel()[idx] * 1000
    pinn_dra = d['pinn_dra'].ravel()[idx] * 1000
    pinn_dde = d['pinn_dde'].ravel()[idx] * 1000

    lim = float(np.nanpercentile(np.abs(np.r_[hgp_dra, hgp_dde, pinn_dra, pinn_dde]), 99))
    lim = max(lim, 1.0)

    fig, axes = plt.subplots(1, 2, figsize=(8, 4.2), constrained_layout=True)
    for ax, hgp_val, pinn_val, name in [
        (axes[0], hgp_dra, pinn_dra, r'$\Delta$RA*'),
        (axes[1], hgp_dde, pinn_dde, r'$\Delta$Dec'),
    ]:
        ax.scatter(pinn_val, hgp_val, s=3, alpha=0.25, rasterized=True)
        ax.plot([-lim, lim], [-lim, lim], color='black', lw=1)
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
        ax.set_xlabel(f'PINN {name} [mas]')
        ax.set_ylabel(f'HGP super-tight {name} [mas]')
        ax.set_title(name)
    fig.suptitle(f'{BAND_LABEL[PLOT_BAND_ST_SCATTER]} vector-component agreement (super-tight HGP vs PINN, raw)')


## 8. Lattice-smoothed HGP vs PINN

Section 7c showed that the residual criss-cross in super-tight HGP is a finite-rank basis-function lattice, not data structure (anchor density does not track the ridges). The pattern wavelength is set by the basis spacing (300 arcsec). Smoothing the HGP field with a Gaussian of sigma = 75 arcsec (about 1/4 the basis spacing) removes the lattice without touching the genuine degree-scale concordance signal — anything coherent on PINN's length scale is preserved; the inter-center ridges are averaged out.

This is a cosmetic transform of HGP, not a refit. It tests the claim that the visible HGP-PINN disagreement in the maps is geometric, not real: if the smoothed HGP map and the per-component scatter both look much closer to PINN, the residual ~2 mas was lattice; if not, the disagreement is intrinsic.


In [ ]:
from scipy.ndimage import gaussian_filter

SMOOTH_SIGMA_PIX = 15  # 75 arcsec at 5 arcsec/pixel (~ 1/4 basis spacing)


def smooth_masked_field(dra, dde, mask, sigma_pix=SMOOTH_SIGMA_PIX):
    valid = mask.astype(np.float64)
    norm = np.maximum(gaussian_filter(valid, sigma_pix), 1e-6)
    sm_dra = gaussian_filter(np.where(mask, dra, 0.0), sigma_pix) / norm
    sm_dde = gaussian_filter(np.where(mask, dde, 0.0), sigma_pix) / norm
    return sm_dra, sm_dde


if have_raw_hgp_supertight():
    rows_sm = []
    for band in BANDS:
        d = load_resampled_band_supertight(band)
        m = support_mask_supertight(d)
        sm_dra, sm_dde = smooth_masked_field(d['hgp_dra'], d['hgp_dde'], m)
        rows_sm.append({
            'band': BAND_LABEL[band],
            'n_support': int(m.sum()),
            'hgp_smooth_rms_mas': rms_amp_mas(sm_dra, sm_dde, m),
            'hgp_st_rms_mas': rms_amp_mas(d['hgp_dra'], d['hgp_dde'], m),
            'pinn_rms_mas': rms_amp_mas(d['pinn_dra'], d['pinn_dde'], m),
            'smooth_minus_pinn_rms_mas': rms_amp_mas(sm_dra - d['pinn_dra'], sm_dde - d['pinn_dde'], m),
            'st_minus_pinn_rms_mas': rms_amp_mas(d['hgp_dra'] - d['pinn_dra'], d['hgp_dde'] - d['pinn_dde'], m),
        })
    compare_sm_df = pd.DataFrame(rows_sm)
    display(compare_sm_df.round(3))


In [ ]:
if have_raw_hgp_supertight():
    PLOT_BAND_SM = 'I'
    d = load_resampled_band_supertight(PLOT_BAND_SM)
    m = support_mask_supertight(d)
    sm_dra, sm_dde = smooth_masked_field(d['hgp_dra'], d['hgp_dde'], m)

    panels = {
        'HGP super-tight, lattice-smoothed (75")': amp_mas(sm_dra, sm_dde),
        'PINN raw |F|':                              amp_mas(d['pinn_dra'], d['pinn_dde']),
    }
    vmax = float(np.nanpercentile(
        np.concatenate([np.where(m, v, np.nan).ravel() for v in panels.values()]),
        98,
    ))
    fig, axes = plt.subplots(1, 2, figsize=(8, 5.6), constrained_layout=True)
    for ax, (title, image) in zip(axes, panels.items()):
        show = np.asarray(image, dtype=np.float64).copy()
        show[~m] = np.nan
        im = ax.imshow(show, origin='lower', cmap='magma', vmin=0, vmax=vmax)
        ax.set_title(title)
        ax.set_xticks([]); ax.set_yticks([])
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02, label='mas')
    fig.suptitle(f'{BAND_LABEL[PLOT_BAND_SM]}: lattice-smoothed HGP vs PINN, raw anchors', y=1.03)
